# Tutorial 08 — Stress-Driven Volumetric Growth

This notebook computes results directly from the local package and does not load committed figures.

In [ ]:
LANGUAGE = "en"
from pathlib import Path
import sys


def _find_repository_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Repository root not found. Open the notebook inside the repository.")


REPOSITORY_ROOT = _find_repository_root(Path.cwd().resolve())
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import biomechanics_tutorials
print("Repository:", REPOSITORY_ROOT)
print("Package:", Path(biomechanics_tutorials.__file__).resolve())

## Imports and local repository bootstrap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from biomechanics_tutorials.growth_kinematics import GrowthMaterialParameters
from biomechanics_tutorials.volumetric_growth import (
    VolumetricGrowthLawParameters,
    evaluate_growth_stimulus,
    growth_response,
    simulate_prescribed_deformation,
    simulate_prescribed_mean_stress,
    simulate_spatial_growth_field,
)

MATERIAL = GrowthMaterialParameters(shear_modulus=1.0, bulk_modulus=18.0)

## Compare possible growth stimuli

In [ ]:
stretch = np.linspace(0.9, 1.12, 120)
measures = ["mean_cauchy", "mean_mandel", "pressure", "energy"]
for measure in measures:
    values = [evaluate_growth_stimulus(s * np.eye(3), 1.0, measure, MATERIAL)["stimulus"] for s in stretch]
    values = np.asarray(values)
    plt.plot(stretch, values / max(np.max(np.abs(values)), 1e-12), label=measure)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("stretch")
plt.ylabel("normalized stimulus")
plt.legend()
plt.show()

## Constitutive response functions

In [ ]:
error = np.linspace(-2, 2, 300)
for dead_zone, limit, label in [(0, None, "linear"), (0.15, None, "dead zone"), (0, 0.7, "saturation"), (0.15, 0.7, "both")]:
    law = VolumetricGrowthLawParameters(dead_zone=dead_zone, response_limit=limit)
    plt.plot(error, [growth_response(e, law) for e in error], label=label)
plt.xlabel("normalized error")
plt.ylabel("response")
plt.legend()
plt.show()

## Fixed-deformation relaxation

In [ ]:
time = np.linspace(0, 18, 361)
F = 1.11 * np.eye(3)
initial = evaluate_growth_stimulus(F, 1.0, "mean_mandel", MATERIAL)["stimulus"]
law = VolumetricGrowthLawParameters(rate=0.18, target=0.0, scale=max(abs(initial), 1.0), measure="mean_mandel", response_limit=2.0)
relaxation = simulate_prescribed_deformation(time, F, material=MATERIAL, law=law)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(time, relaxation["Jg"])
axes[0].set_ylabel("Jg")
axes[1].plot(time, relaxation["mean_mandel"], label="mean Mandel")
axes[1].plot(time, relaxation["von_mises"], label="von Mises")
axes[1].legend()
plt.show()

## Boundary control changes the interpretation

In [ ]:
stress_law = VolumetricGrowthLawParameters(rate=0.18, target=0.0, scale=1.0, measure="mean_cauchy", response_limit=2.0)
stress_control = simulate_prescribed_mean_stress(time, 0.45, material=MATERIAL, law=stress_law)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(time, stress_control["stimulus"])
axes[0].set_ylabel("controlled stress")
axes[1].plot(time, stress_control["Jg"], label="Jg")
axes[1].plot(time, stress_control["total_stretch"], label="total stretch")
axes[1].legend()
plt.show()

## Hydrostatic target does not remove deviatoric stress

In [ ]:
uniaxial = np.diag([1.25, 1.0, 1.0])
initial_u = evaluate_growth_stimulus(uniaxial, 1.0, "mean_mandel", MATERIAL)["stimulus"]
law_u = VolumetricGrowthLawParameters(rate=0.22, target=0.0, scale=max(abs(initial_u), 1.0), measure="mean_mandel", response_limit=2.0)
result_u = simulate_prescribed_deformation(np.linspace(0, 25, 501), uniaxial, material=MATERIAL, law=law_u)
print("Final mean Mandel:", result_u["mean_mandel"][-1])
print("Final von Mises:", result_u["von_mises"][-1])

## Spatial heterogeneity and regularization

In [ ]:
x = np.linspace(0, 1, 61)
t = np.linspace(0, 3, 121)
target = 0.18 * np.sin(8*np.pi*x)
spatial_law = VolumetricGrowthLawParameters(rate=0.25, target=0.0, scale=1.0, measure="mean_mandel", response_limit=2.0)
local = simulate_spatial_growth_field(t, x, 1.07*np.eye(3), np.ones_like(x), target, spatial_law, MATERIAL, diffusivity=0.0)
smooth = simulate_spatial_growth_field(t, x, 1.07*np.eye(3), np.ones_like(x), target, spatial_law, MATERIAL, diffusivity=3e-4)
plt.plot(x, local["Jg"][-1], label="local")
plt.plot(x, smooth["Jg"][-1], label="regularized")
plt.xlabel("position")
plt.ylabel("Jg")
plt.legend()
plt.show()

## Minimal verification checklist

In [ ]:
checks = {
    "Jg positive": bool(np.all(relaxation["Jg"] > 0)),
    "mean stress relaxed": bool(abs(relaxation["mean_mandel"][-1]) < 0.05 * abs(relaxation["mean_mandel"][0])),
    "stress control preserved stimulus": bool(np.allclose(stress_control["stimulus"], 0.45)),
    "deviatoric stress remains": bool(result_u["von_mises"][-1] > 0.1),
}
checks